# 02 Fine-Tuning with LoRA (Interactive)

This notebook fine-tunes `google/flan-t5-base` for summarization using LoRA adapters.
Training is configured to balance performance and limited GPU resources.

In [1]:
# experiment configuration
import sys
import random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

_ = (
    load_dataset,
    LoraConfig,
    TaskType,
    get_peft_model,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

print(f"Device: {DEVICE}")

Device: cuda


In [2]:
# experiment configuration
MODEL_NAME = "google/flan-t5-base"
DATASET_NAME = "cnn_dailymail"
DATASET_CONFIG = "3.0.0"
OUTPUT_DIR = ROOT / "lora-adapter"

MAX_TRAIN_SAMPLES = 5000
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LOGGING_STEPS = 20
SAVE_STEPS = 250

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q", "v"]

USE_WANDB = False
WANDB_PROJECT = "domain-summarizer"
RUN_NAME = f"flan-t5-lora-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"

print("Configuration ready.")
print(f"Samples: {MAX_TRAIN_SAMPLES} | Epochs: {NUM_EPOCHS}")

Configuration ready.
Samples: 5000 | Epochs: 1


In [3]:
# optional Weights & Biases initialization
if USE_WANDB:
    import wandb

    wandb.init(
        project=WANDB_PROJECT,
        name=RUN_NAME,
        config={
            "model_name": MODEL_NAME,
            "max_train_samples": MAX_TRAIN_SAMPLES,
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS,
            "batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "seed": SEED,
        },
    )
    print("W&B run started.")
else:
    print("W&B logging disabled.")

W&B logging disabled.


In [4]:
# load tokenizer and base model
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        dtype=DTYPE,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    print("Base model loaded successfully.")
except Exception as exc:
    raise RuntimeError(
        "Failed to load base model/tokenizer. Check internet, model access, and available memory."
    ) from exc

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Base model loaded successfully.


In [5]:
# configure and attach LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA adapters attached.")

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096
LoRA adapters attached.


In [6]:
# load and preprocess training subset
train_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
train_ds = train_ds.shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))


def preprocess(batch):
    prompts = [f"Summarize the following article:\n\n{x}" for x in batch["article"]]
    model_inputs = tokenizer(
        prompts,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["highlights"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train subset",
)

print(tokenized_train)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5000
})


In [7]:
# build trainer components
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=1,
    fp16=(DEVICE == "cuda"),
    bf16=False,
    report_to=["wandb"] if USE_WANDB else [],
    seed=SEED,
    dataloader_pin_memory=(DEVICE == "cuda"),
    remove_unused_columns=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=collator,
)

print("Trainer ready.")

Trainer ready.


In [9]:
# optional close W&B run
if USE_WANDB:
    import wandb
    wandb.finish()
    print("W&B run finished.")
else:
    print("No W&B run to close.")

No W&B run to close.


In [11]:
# cleanup
model = model.cpu()
del trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup complete.")

Cleanup complete.
